**1) Importar librerías**

In [13]:
import re, time, os, pathlib
import pandas as pd
import requests
import numpy as np
from astropy.io import fits
from skimage.feature import canny
from skimage.filters import unsharp_mask
from sklearn.decomposition import PCA
from astropy.visualization import make_lupton_rgb, ZScaleInterval, LogStretch, MinMaxInterval
from concurrent.futures import ThreadPoolExecutor, as_completed

**2) Configuración**

In [14]:
# ---------- CONFIG GLOBAL ----------
LAYER     = "ls-dr10"   # capa DR10
PIXSCALE  = 0.262       # arcsec/pix
SIZE      = 224         # lado (pix)
TIMEOUT_S = 60
WORKERS   = 8
SLEEP_S   = 0.05
BANDS     = "gri"      # DR10 soporta g,r,i,z
TRANSFORM = "rgi_canny_stack"  # "rgi_stack", "rgi_lognorm_stack", "rgi_unsharp_mask", "pca_stack" or "rgi_canny_stack"

OUTPUT_DIR = pathlib.Path(f"ls_dr10_{SIZE}_{BANDS}_{PIXSCALE*1000:.0f}_{TRANSFORM}")    # dr#_imgz_bands_scale
DATA_DIR   = OUTPUT_DIR / "data"            # carpeta única para todos los FITS
OUTPUT_CSV = "lsdr10_paths.csv"
# -----------------------------------

DATA_DIR.mkdir(parents=True, exist_ok=True)

**3) Funciones helper**

In [ ]:
# --- para SDSS-DR14 (columna 'anillos') ---
ANILLOS_LABEL = {
    0:  "none",
    4:  "inner",
    8:  "outer",
    12: "inner+outer",
}

# crear carpeta de salida por cada clase
for label in ANILLOS_LABEL.values():
    (DATA_DIR / label).mkdir(parents=True, exist_ok=True)

def to_int(x, default=0):
    """Convierte un valor a entero.

    Args:
        x (float | str | None): Valor a convertir.
        default (int, optional): Valor por defecto en caso de error. Defaults to 0.

    Returns:
        int: Valor convertido o el valor por defecto.
    """
    try:
        return int(x)
    except Exception:
        return default

def ring_type_from_code(code: int) -> str:
    """Convierte un código de anillo a su etiqueta correspondiente.

    Args:
        code (int): Código del anillo.

    Returns:
        str: Etiqueta del anillo o "unknown" si el código no es válido.
    """
    return ANILLOS_LABEL.get(code, "unknown")

# --- cutout URL ---
def fits_url(ra, dec, size=SIZE, bands=BANDS, layer=LAYER, pixscale=PIXSCALE):
    return (f"https://www.legacysurvey.org/viewer/fits-cutout"
            f"?ra={ra:.8f}&dec={dec:.8f}&layer={layer}"
            f"&pixscale={pixscale:.6f}&size={size}&bands={bands}")


class Transformations:
    @staticmethod
    def log_n_scale_transform(img_data, log_a=1000, clip_mode='minmax', med_val=20):
        """
        Aplicar la transformación Log-N a una imagen usando escala Zscale o Min/Max.

        Pasos del proceso:
        1. Calcular los límites de intensidades.
        2. Normalizar los valores a un rango de 0 a 1.
        3. Aplicar la transformación Log-N.
        4. Escalar la imagen de vuelta al rango original.
        5. Modificar la mediana para mantener consistencia.

        Args:
        img_data (np.ndarray): Imagen de entrada.
        log_a (float): Factor de escala para la transformación Log-N.
        clip_mode (str): Modo de escala ('zscale' o 'minmax').

        Returns:
            np.ndarray: Imagen transformada.
        """
        img = img_data.astype(np.float64)

        # 1. Calcular los límites de intensidades
        if clip_mode == 'zscale':
            interval = ZScaleInterval()
        elif clip_mode == 'minmax':
            interval = MinMaxInterval()
        p_low, p_high = interval.get_limits(img)

        # Manejar imagen plana
        if p_high <= p_low:
            return np.zeros_like(img, dtype=np.uint8)

        # 2. Normalizar los valores de pixeles entre 0-1
        x = (img - p_low) / (p_high - p_low)
        x = np.clip(x, 0.0, 1.0)

        # 3. Aplicar la transformación log
        log_stretch = LogStretch(a=log_a)
        y = log_stretch(x)

        # 4. Escalar la imagen de vuelta al rango original
        display_image = (y * 255.0).astype(np.uint8)

        # 5. Ajustar la mediana para constancia
        if med_val:
            median_val = np.median(display_image)
            rendered_img = np.clip(display_image - median_val + med_val, 0, 255).astype(np.uint8)
        else:
            rendered_img = display_image

        return rendered_img

    @staticmethod
    def rgi_stack(channel_r, channel_g, channel_i):
        """ Aplicar transformaciones a una imagen RGB.
        Args:
            channel_r (np.ndarray): Canal R.
            channel_g (np.ndarray): Canal G.
            channel_i (np.ndarray): Canal I.

        Returns:
            np.ndarray: Imagen RGB transformada.
        """
        rgb_img = make_lupton_rgb(channel_r, channel_g, channel_i, stretch=0.5, Q=8)
        return rgb_img
    
    @staticmethod
    def rgi_lognorm_stack(channel_r, channel_g, channel_i):
        """ Aplicar transformaciones LogNorm a un stack.
        Args:
            channel_r (np.ndarray): Canal R.
            channel_g (np.ndarray): Canal G.
            channel_i (np.ndarray): Canal I.

        Returns:
            np.ndarray: Imagen transformada.
        """
        rgb_img = np.stack([Transformations.log_n_scale_transform(channel_r),
                            Transformations.log_n_scale_transform(channel_g),
                            Transformations.log_n_scale_transform(channel_i)],
                            axis=-1)
        rgb_img = (np.clip(rgb_img, 0, 255)).astype(np.uint8)

        return rgb_img
    
    @staticmethod
    def rgi_unsharp_mask(channel_r, channel_g, channel_i):
        """Aplicar transformaciones Unsharp Masking a un stack.
        Args:
            channel_r (np.ndarray): Canal R.
            channel_g (np.ndarray): Canal G.
            channel_i (np.ndarray): Canal I.

        Returns:
            np.ndarray: Imagen transformada.
        """
        channel_r = Transformations.log_n_scale_transform(channel_r)
        channel_g = Transformations.log_n_scale_transform(channel_g)
        channel_i = Transformations.log_n_scale_transform(channel_i)

        r_usm = unsharp_mask(channel_r, radius=15, amount=1.5, preserve_range=False)
        g_usm = unsharp_mask(channel_g, radius=15, amount=1.5, preserve_range=False)
        i_usm = unsharp_mask(channel_i, radius=15, amount=1.5, preserve_range=False)

        rgb = np.stack([r_usm, g_usm, i_usm], axis=-1)
        rgb8 = (np.clip(rgb, 0, 255) * 255.0).astype(np.uint8)

        return rgb8
    
    @staticmethod
    def pca_stack(channel_r, channel_g, channel_i):
        """
        Aplica PCA a un stack multicanal (R, G, I) y retorna métricas clave.

        Args:
            channel_r, channel_g, channel_i (np.ndarray): Canales individuales.
            n_components (int): Número de componentes PCA.

        Returns:
            dict: {'var_ratio', 'cum_var', 'vector_resumen', 'snr_pca'}
        """
        channel_r = Transformations.log_n_scale_transform(channel_r)
        channel_g = Transformations.log_n_scale_transform(channel_g)
        channel_i = Transformations.log_n_scale_transform(channel_i)
        # Transformar cada imagen con canales G, R, I
        canales_transformados = [channel_r, channel_g, channel_i]
        # Aplanar para que quede HxWxC=# y apilar para PCA
        vectores = [canal.flatten() for canal in canales_transformados]
        matriz = np.stack(vectores, axis=0).T  # (H×W, 3)

        #n_valid asegura que no se pidan más componentes de los que se pueden calcular
        n_samples, n_features = matriz.shape
        n_valid = min(3, n_samples, n_features)

        #PCA
        pca = PCA(n_components=n_valid)
        pca_resultado = pca.fit_transform(matriz)
        
        pca_1 = pca_resultado[:, 0]
        img_pca1 = pca_1.reshape(channel_r.shape)
        img_norm = (img_pca1 - np.min(img_pca1)) / (np.max(img_pca1) - np.min(img_pca1))
        img_uint8 = (img_norm * 255).astype(np.uint8)

        return img_uint8
    
    @staticmethod
    def rgi_canny_stack(channel_r, channel_g, channel_i):
        """ Aplicar transformaciones Canny a un stack.
        Args:
            channel_r (np.ndarray): Canal R.
            channel_g (np.ndarray): Canal G.
            channel_i (np.ndarray): Canal I.

        Returns:
            np.ndarray: Imagen transformada.
        """
        # Preparar imagen en escala de grises a partir de G, R, I
        r8 = Transformations.log_n_scale_transform(channel_r).astype(np.float64)
        g8 = Transformations.log_n_scale_transform(channel_g).astype(np.float64)
        i8 = Transformations.log_n_scale_transform(channel_i).astype(np.float64)
        gray = (g8 + r8 + i8) / (3.0 * 255.0)  # normalizar a [0, 1]

        # Aplicar canny
        edges = canny(gray, sigma=1, low_threshold=0.50, high_threshold=0.90, use_quantiles=True)

        return edges

**4) SDSS-DR14 (Descarga de FITS, bandas gri y transformación)**

In [ ]:
def fetch_one_sdss(row, outdir: pathlib.Path):
    """Descarga un FITS de SDSS basado en la información de una fila del DataFrame.
    
    Args:
        row (dict): Fila del DataFrame con las columnas 'ra', 'dec', 'objID', 'anillos'.
        outdir (pathlib.Path): Directorio donde se guardará el archivo FITS.

    Returns:
        dict: Diccionario con las claves 'fits_url', 'fits_path', 'status'.
    """
    # row debe tener: ra, dec, objID, anillos
    if pd.isna(row["ra"]) or pd.isna(row["dec"]):
        return {"fits_url": None, "fits_path": None, "status": "skip_nan"}
    ra   = float(row["ra"])
    dec  = float(row["dec"])
    oid  = str(row.get("objID", f"row{row.get('_idx', 0)}"))
    code = to_int(row.get("anillos", 0))
    rtype = ring_type_from_code(code)

    # Nombre de archivo a partir de RA, DEC, objID, tipo de anillo
    base = f"SDSS_{oid}_{rtype}_ra{ra:.6f}_dec{dec:.6f}_s{SIZE}"
    dest = outdir / rtype / f"{base}.png"
    url  = fits_url(ra, dec)
    print(url)

    status = "ok"

    try:
        # Descargar si no existe o está vacío
        if not dest.exists() or dest.stat().st_size == 0:
            r = requests.get(url, timeout=TIMEOUT_S)
            print(f"[SDSS] Procesando {dest}...")
            if r.ok and r.content:  # Si la respuesta es válida, guardar el contenido
                # leer el FITS
                with fits.open(r.content) as hdul:
                    data = hdul[0].data
                    r_band = data[1]
                    g_band = data[0]
                    i_band = data[2]
                    # Aplicar la transformación seleccionada
                    if TRANSFORM == "rgi_stack":
                        img = Transformations.rgi_stack(r_band, g_band, i_band)
                    elif TRANSFORM == "rgi_lognorm_stack":
                        img = Transformations.rgi_lognorm_stack(r_band, g_band, i_band)
                    elif TRANSFORM == "rgi_unsharp_mask":
                        img = Transformations.rgi_unsharp_mask(r_band, g_band, i_band)
                    elif TRANSFORM == "pca_stack":
                        img = Transformations.pca_stack(r_band, g_band, i_band)
                    elif TRANSFORM == "rgi_canny_stack":
                        edges = Transformations.rgi_canny_stack(r_band, g_band, i_band)
                        img = np.stack([edges]*3, axis=-1).astype(np.uint8) * 255
                    else:
                        status = "error:unknown_transform"
                        img = None

                    if img is not None:
                        from PIL import Image
                        im = Image.fromarray(img)
                        im.save(dest)
                    else:
                        status = "error:processing_failed"
            else:   # Respuesta no válida, registrar el estado del error
                status = f"http_{r.status_code}"
    except Exception as e:
        status = f"error:{e.__class__.__name__}"

    time.sleep(SLEEP_S)
    return {"fits_url": url, "fits_path": str(dest), "status": status}


# ---------------- PRINCIPAL ----------------
sdss_out_df = None

df_sdss = pd.read_csv("..\data\dataset.csv")

# Incluir solo filas con datos de ANILLOS_LABEL
df_sdss = df_sdss[df_sdss["anillos"].isin(ANILLOS_LABEL.keys())].copy()
# imprimir el conteo por tipo de anillo
print(f"[SDSS] Conteo por tipo de anillo:\n{df_sdss['anillos'].map(ANILLOS_LABEL).value_counts()}")

for col in ["ra", "dec", "anillos"]:
    if col not in df_sdss.columns:
        raise SystemExit(f"[SDSS] Falta la columna '{col}'. Presentes: {list(df_sdss.columns)[:20]}")

# columnas derivadas
df_sdss["anillos"]    = df_sdss["anillos"].apply(to_int)
df_sdss["ring_type"]  = df_sdss["anillos"].apply(ring_type_from_code)
df_sdss["is_partial"] = (df_sdss["anillos"] == 16)
df_sdss["_idx"] = range(len(df_sdss))

# descargas
rows = df_sdss.to_dict(orient="records")
downloads = []
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = [ex.submit(fetch_one_sdss, row, DATA_DIR) for row in rows]
    for f in as_completed(futs):
        downloads.append(f.result())

dl_df = pd.DataFrame(downloads)

assert len(dl_df) == len(df_sdss), "[SDSS] #descargas != #filas"

sdss_out_df = pd.concat([df_sdss.reset_index(drop=True),
                        pd.DataFrame(dl_df, columns=["fits_url","fits_path","status"])], axis=1)
sdss_out_df["source"] = "sdss_anillos"

sdss_out_df.to_csv(OUTPUT_DIR / OUTPUT_CSV, index=False)
print(f"[SDSS] Filas: {len(sdss_out_df)}, OK: {(sdss_out_df.status=='ok').sum()}")

ok = (sdss_out_df.status == "ok").sum()
print(f"Guardado CSV: {OUTPUT_DIR / OUTPUT_CSV}")
print(f"Imágenes en: {DATA_DIR}")
print(f"Descargas OK: {ok} / {len(sdss_out_df)}")
print("\nConteo por ring_type:")
print(sdss_out_df["ring_type"].value_counts(dropna=False))

[SDSS] Conteo por tipo de anillo:
anillos
none           6660
inner           857
inner+outer     372
outer           186
Name: count, dtype: int64
https://www.legacysurvey.org/viewer/fits-cutout?ra=134.44717000&dec=-0.19997270&layer=ls-dr10&pixscale=0.262000&size=224&bands=gri
https://www.legacysurvey.org/viewer/fits-cutout?ra=198.23356000&dec=0.94118815&layer=ls-dr10&pixscale=0.262000&size=224&bands=gri
https://www.legacysurvey.org/viewer/fits-cutout?ra=199.29492000&dec=0.52757058&layer=ls-dr10&pixscale=0.262000&size=224&bands=gri
https://www.legacysurvey.org/viewer/fits-cutout?ra=165.74061000&dec=-0.96209487&layer=ls-dr10&pixscale=0.262000&size=224&bands=gri
https://www.legacysurvey.org/viewer/fits-cutout?ra=31.37202000&dec=13.25101600&layer=ls-dr10&pixscale=0.262000&size=224&bands=gri
https://www.legacysurvey.org/viewer/fits-cutout?ra=29.48267200&dec=13.35815900&layer=ls-dr10&pixscale=0.262000&size=224&bands=gri
https://www.legacysurvey.org/viewer/fits-cutout?ra=199.54195000&dec=-1

**4) SDSS-DR14 (Creación de dataset con archivos guardados en local)**

In [16]:
import matplotlib.pyplot as plt
import cv2

for i, img in enumerate(pathlib.Path("C:/MNA/MNA-V/Proyecto Integrador/notebooks/ls_dr10_224_gri_262/fits").glob("*.fits")):
    ring_type = img.name.split("_")[-4]
    with fits.open(img) as hdul:
        data = hdul[0].data
        r_band = data[1]
        g_band = data[0]
        i_band = data[2]
        # Aplicar la transformación seleccionada
        if TRANSFORM == "rgi_stack":
            img_trans = Transformations.rgi_stack(r_band, g_band, i_band)
        elif TRANSFORM == "rgi_lognorm_stack":
            img_trans = Transformations.rgi_lognorm_stack(r_band, g_band, i_band)
        elif TRANSFORM == "rgi_unsharp_mask":
            img_trans = Transformations.rgi_unsharp_mask(r_band, g_band, i_band)
        elif TRANSFORM == "pca_stack":
            img_trans = Transformations.pca_stack(r_band, g_band, i_band)
        elif TRANSFORM == "rgi_canny_stack":
            edges = Transformations.rgi_canny_stack(r_band, g_band, i_band)
            img_trans = np.stack([edges]*3, axis=-1).astype(np.uint8) * 255
        else:
            status = "error:unknown_transform"
            img_trans = None
        print(str(OUTPUT_DIR / ring_type / img.name.replace(".fits", ".png")))
        if img_trans.shape[-1] != 3:
            check = cv2.imwrite(str(DATA_DIR / ring_type / img.name.replace(".fits", ".png")), cv2.cvtColor(img_trans, cv2.COLOR_GRAY2BGR))
        else:
            check = cv2.imwrite(str(DATA_DIR / ring_type / img.name.replace(".fits", ".png")), cv2.cvtColor(img_trans, cv2.COLOR_RGB2BGR))
    print(i)
    if not check:
        break

ls_dr10_224_gri_262_rgi_canny_stack\outer\SDSS_1237648672921485632_outer_ra243.708880_dec-0.915654_s224.png
0
ls_dr10_224_gri_262_rgi_canny_stack\none\SDSS_1237648673994965546_none_ra243.236760_dec-0.096260_s224.png
1
ls_dr10_224_gri_262_rgi_canny_stack\none\SDSS_1237648673997127724_none_ra248.064170_dec-0.049930_s224.png
2
ls_dr10_224_gri_262_rgi_canny_stack\none\SDSS_1237648675606430099_none_ra245.086640_dec1.228430_s224.png
3
ls_dr10_224_gri_262_rgi_canny_stack\none\SDSS_1237648702966988820_none_ra184.535660_dec-1.064119_s224.png
4
ls_dr10_224_gri_262_rgi_canny_stack\none\SDSS_1237648702972297393_none_ra196.773530_dec-1.176851_s224.png
5
ls_dr10_224_gri_262_rgi_canny_stack\none\SDSS_1237648702972625038_none_ra197.421180_dec-1.048272_s224.png
6
ls_dr10_224_gri_262_rgi_canny_stack\none\SDSS_1237648702973542469_none_ra199.541950_dec-1.243647_s224.png
7
ls_dr10_224_gri_262_rgi_canny_stack\none\SDSS_1237648702975180927_none_ra203.229690_dec-1.159490_s224.png
8
ls_dr10_224_gri_262_rgi_can